# 🤖 Agent Development - Risk Analyst & Compliance Officer

Welcome to Phase 2 & 3 of the project! In this notebook, you'll develop and test both AI agents:

1. **Risk Analyst Agent** (Chain-of-Thought prompting)
2. **Compliance Officer Agent** (ReACT prompting)

## 🎯 Learning Objectives
- Implement Chain-of-Thought prompting for systematic reasoning
- Build ReACT prompting for structured action-taking
- Handle structured JSON outputs from LLMs
- Create robust error handling and validation
- Test agents with real financial data

## 🚀 Setup and Environment

In [1]:
# Import required libraries
import os
import sys
import json
import openai
from dotenv import load_dotenv
from datetime import datetime

# Add src directory to Python path for module imports
sys.path.append(os.path.abspath('../src'))

# Load environment variables
load_dotenv('../.env')

print("📚 Libraries loaded!")
print("🔐 Environment variables loaded")
print("📂 Source directory added to Python path:", os.path.abspath('../src'))

📚 Libraries loaded!
🔐 Environment variables loaded
📂 Source directory added to Python path: c:\Users\sudip\Projects\courses\Udacity_Learning_Notes\AgenticAI_Finance\Project_TRACE\starter\src


In [2]:
# OpenAI Setup
import openai

openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    print("⚠️ WARNING: No OpenAI API key found!")
    print("Please set OPENAI_API_KEY in your .env file")
else:
    # Vocareum requires routing through their servers
    client = openai.OpenAI(
        api_key=openai_api_key
    )
    print("✅ OpenAI client initialized")
    print(f"🔑 API key: {openai_api_key[:8]}...{openai_api_key[-4:]}")


✅ OpenAI client initialized
🔑 API key: sk-proj-...kCIA


## 📊 Phase 1 Review: Load Foundation Components

Before building agents, let's ensure your foundation components are working.

In [3]:
# Import your implemented foundation components

from foundation_sar import (
    CustomerData,
    AccountData,
    TransactionData,
    CaseData,
    RiskAnalystOutput,
    ComplianceOfficerOutput,
    ExplainabilityLogger,
    DataLoader,
    load_csv_data
)

print("✅ Foundation components imported, to create sample cases for agent testing")

✅ Foundation components imported, to create sample cases for agent testing


In [4]:
# Testing foundation components
# Having implemented foundation_sar.py, this cell is used to:
# 1. Load CSV data
# 2. Create a DataLoader instance
# 3. Generate a sample case for testing agents

import pandas as pd

print("📋 Testing foundation components")

print("1. Load CSV data from ../data/")
customers_df, accounts_df, transactions_df = load_csv_data("../data/")

print("2. Create ExplainabilityLogger instance")
logger = ExplainabilityLogger("../outputs/audit_logs/agent_development.jsonl")

print("3. Create DataLoader instance")
loader = DataLoader(logger)

print("4. Generate sample case for agent testing")
sample_customer = customers_df[customers_df["customer_id"] == "CUST_0002"].copy()
sample_customer['ssn_last_4'] = sample_customer['ssn_last_4'].astype(str)
sample_account = accounts_df[accounts_df["customer_id"] == "CUST_0002"].copy()
sample_transaction = transactions_df[transactions_df["account_id"].isin(sample_account["account_id"])].copy()
sample_transaction["transaction_date"] = pd.to_datetime(
    sample_transaction["transaction_date"]
).dt.strftime("%Y-%m-%d")
sample_transaction["counterparty"] = (
    sample_transaction["counterparty"].fillna("")
)
sample_transaction['location'] = sample_transaction['location'].astype(str)

sample_customer_dict = sample_customer.to_dict(orient="records")[0]
sample_account_dict = sample_account.to_dict(orient="records")
sample_transaction_dict = sample_transaction.to_dict(orient="records")

sample_case = loader.create_case_from_data(sample_customer_dict, sample_account_dict, sample_transaction_dict)

print("5. Printing the sample case")

print(f" Sample Case Type: {type(sample_case)}; Sample Case: {sample_case}")



📋 Testing foundation components
1. Load CSV data from ../data/
2. Create ExplainabilityLogger instance
3. Create DataLoader instance
4. Generate sample case for agent testing
5. Printing the sample case
 Sample Case Type: <class 'foundation_sar.CaseData'>; Sample Case: case_id='879aa4ae-21c5-4e44-bcb1-b3ef9a1f9efc' customer=CustomerData(customer_id='CUST_0002', name='Renee Blair', date_of_birth='1978-05-14', ssn_last_4='7201', address='959 Janet Cape Apt. 413, South Joshuastad, GA 49021', customer_since='2023-03-03', risk_rating=<RiskAssessment.LOW: 'Low'>, phone='(534)719-2832x764', occupation='Banker', annual_income=72005) accounts=[AccountData(account_id='CUST_0002_ACC_1', customer_id='CUST_0002', account_type='Money_Market', opening_date='2023-12-03', current_balance=92121.66, average_monthly_balance=172242.96, status='Active')] transactions=[TransactionData(transaction_id='TXN_B24455F3', account_id='CUST_0002_ACC_1', transaction_date='2025-02-18', transaction_type='Online_Transfer

## 🔍 Phase 2: Risk Analyst Agent Development

The Risk Analyst Agent uses **Chain-of-Thought prompting** to systematically analyze suspicious activity patterns.

### 📚 Understanding Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting guides AI models through step-by-step reasoning:

1. **Explicit Steps**: Break complex reasoning into clear phases
2. **Sequential Logic**: Each step builds on previous ones
3. **Domain Expertise**: Frame AI as subject matter expert
4. **Structured Output**: Guide toward specific response format

In [5]:
import os
import sys

print("Current directory:", os.getcwd())
print("Python executable:", sys.executable)
print("Python search paths:", sys.path)

Current directory: c:\Users\sudip\Projects\courses\Udacity_Learning_Notes\AgenticAI_Finance\Project_TRACE\starter\notebooks
Python executable: c:\Users\sudip\Projects\courses\Udacity_Learning_Notes\.venv\Scripts\python.exe
Python search paths: ['C:\\Users\\sudip\\AppData\\Roaming\\uv\\python\\cpython-3.11.14-windows-x86_64-none\\python311.zip', 'C:\\Users\\sudip\\AppData\\Roaming\\uv\\python\\cpython-3.11.14-windows-x86_64-none\\DLLs', 'C:\\Users\\sudip\\AppData\\Roaming\\uv\\python\\cpython-3.11.14-windows-x86_64-none\\Lib', 'C:\\Users\\sudip\\AppData\\Roaming\\uv\\python\\cpython-3.11.14-windows-x86_64-none', 'c:\\Users\\sudip\\Projects\\courses\\Udacity_Learning_Notes\\.venv', '', 'c:\\Users\\sudip\\Projects\\courses\\Udacity_Learning_Notes\\.venv\\Lib\\site-packages', 'c:\\Users\\sudip\\Projects\\courses\\Udacity_Learning_Notes\\AgenticAI_Finance\\Project_TRACE\\starter\\src']


In [6]:
# Test Chain-of-Thought prompt design
# This cell helps you design and test your CoT prompt structure

sys.path.append("..")

from src.risk_analyst_agent import create_chain_of_thought_framework

def design_cot_prompt():
    """Design and test Chain-of-Thought prompt for risk analysis"""

    cot_prompt = create_chain_of_thought_framework()
    
    return cot_prompt["step_1"]

# Test your prompt design
cot_prompt = design_cot_prompt()
print("🧠 Chain-of-Thought Prompt Design:")
print(cot_prompt[:200] + "...")

🧠 Chain-of-Thought Prompt Design:

        Data Review:
        Review all available customer, account, transaction, alert, and case information.

        Extract only factual information relevant to a potential Suspicious Activity Re...


In [7]:
# Implement and test Risk Analyst Agent - SIMPLE SMOKE TEST
from openai import OpenAI
import sys
from pathlib import Path

# Notebook is located in starter/notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.risk_analyst_agent import RiskAnalystAgent

def create_sample_case(cust_id:str):
    # Load CSV data from ../data/
    customers_df, accounts_df, transactions_df = load_csv_data("../data/")

    # Create ExplainabilityLogger instance
    logger = ExplainabilityLogger("../outputs/audit_logs/smoke_test.jsonl")

    # Create DataLoader instance
    loader = DataLoader(logger)

    try:
        # Generate sample case
        sample_customer = customers_df[customers_df["customer_id"] == cust_id].copy()
        sample_customer['ssn_last_4'] = sample_customer['ssn_last_4'].astype(str)
        sample_account = accounts_df[accounts_df["customer_id"] == cust_id].copy()
        sample_transaction = transactions_df[transactions_df["account_id"].isin(sample_account["account_id"])].copy()
        sample_transaction["transaction_date"] = pd.to_datetime(
            sample_transaction["transaction_date"]
        ).dt.strftime("%Y-%m-%d")
        sample_transaction["counterparty"] = (
            sample_transaction["counterparty"].fillna("")
        )
        sample_transaction['location'] = sample_transaction['location'].astype(str)

        sample_customer_dict = sample_customer.to_dict(orient="records")[0]
        sample_account_dict = sample_account.to_dict(orient="records")
        sample_transaction_dict = sample_transaction.to_dict(orient="records")

        sample_case = loader.create_case_from_data(sample_customer_dict, sample_account_dict, sample_transaction_dict)
        return sample_case

    except Exception as e:
        print(f"Error generating sample case: {e}")
        return None

def simple_risk_analyst_smoke_test():
    """
    A simple smoke test for Risk Analyst Agent
    
    This is a basic test that:
    1. Creates a RiskAnalystAgent instance
    2. Creates simple test data (or uses mock data)
    3. Calls analyze_case() method
    4. Verifies the result has the expected structure
    5. Prints success/failure
    
    Keep this simple - just verify your agent doesn't crash and returns something reasonable!
    """
    print("🔍 Risk Analyst Smoke Test")
    logger = ExplainabilityLogger("../outputs/audit_logs/smoke_test.jsonl")
    
    load_dotenv('../../.env')
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    try:
        print("📋 Creating a RiskAnalystAgent instance")
        agent = RiskAnalystAgent(client, logger)
        # Create minimal test case
        test_case = create_sample_case("CUST_0002")
        result = agent.analyze_case(test_case)
        print(f"✅ SUCCESS: Got classification '{result.classification}' with confidence {result.confidence_score}")
        print("✅ Smoke test passed")
    except Exception as e:
        print(f"❌ Smoke test failed: {e}")

#my_case =create_sample_case("CUST_0002")
#print(my_case)
simple_risk_analyst_smoke_test()

🔍 Risk Analyst Smoke Test
📋 Creating a RiskAnalystAgent instance
🚀 STARTING CASE ANALYSIS

Finish reason: stop
Model: gpt-4.1-mini-2025-04-14

Usage: CompletionUsage(completion_tokens=597, prompt_tokens=1295, total_tokens=1892, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None))

========== RAW STEP 1 RESPONSE ==========
{'case_id': '6d27248b-a202-4aba-a189-db044c0d602a', 'customer': {'customer_id': 'CUST_0002', 'name': 'Renee Blair', 'risk_rating': <RiskAssessment.LOW: 'Low'>, 'occupation': 'Banker', 'annual_income': 72005, 'customer_since': '2023-03-03'}, 'accounts': [{'account_id': 'CUST_0002_ACC_1', 'account_type': 'Money_Market', 'status': 'Active', 'current_balance': 92121.66}], 'activity_period': {'start_date': '2025-02-18', 'end_da

### 🧪 Risk Analyst Testing Framework

In [8]:
# COMPREHENSIVE Risk Analyst Testing - Import Pre-Built Test Suite
# Students: Use our comprehensive test suite instead of writing your own

import sys
import os

# Add tests directory to Python path for importing test modules
project_root = os.path.abspath('..')
tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

print(f"📁 Added tests directory to Python path: {tests_path}")

def run_comprehensive_risk_analyst_tests():
    """
    Use pre-built comprehensive test suite to validate your Risk Analyst Agent
    
    These tests validate:
    - Agent initialization and configuration
    - Case analysis with valid inputs
    - JSON parsing and error handling
    - System prompt structure and content
    - API call parameters and responses
    - Helper method functionality
    """
    print("🧪 Comprehensive Risk Analyst Testing")
    print("📋 TODO: Uncomment and run after implementing your Risk Analyst Agent")
    
    # Uncomment when your agent is ready:
    try:
        from test_risk_analyst import TestRiskAnalystAgent
        import pytest
        
        print("🔍 Loading comprehensive test suite...")
        
        # Run the test suite
        print("🚀 Running Risk Analyst test suite...")
        result = pytest.main([
            f"{tests_path}/test_risk_analyst.py", 
            "-v", 
            "--tb=short"
        ])
        
        if result == 0:
            print("✅ All Risk Analyst tests passed!")
        else:
            print("❌ Some tests failed. Check the output above for details.")
            
    except ImportError as e:
        print(f"❌ Import Error: {e}")

# Quick preview of available tests
try:
    from test_risk_analyst import TestRiskAnalystAgent
    import inspect
    
    # Get all test methods
    test_methods = [method for method in dir(TestRiskAnalystAgent) 
                   if method.startswith('test_')]
    
    print("📊 Preview of Comprehensive Risk Analyst Tests:")
    for method_name in test_methods[:5]:  # Show first 5
        method = getattr(TestRiskAnalystAgent, method_name)
        doc = method.__doc__ or method_name.replace('_', ' ').title()
        print(f"   • {doc}")
    if len(test_methods) > 5:
        print(f"   ... and {len(test_methods) - 5} more tests")
    print("\n💡 These tests validate edge cases you might not think of!")
    print("💡 Much more thorough than manual testing!")
except Exception as e:
    print(f"ℹ️ Test suite will be available after implementing foundation_sar.py: {e}")

# Call the function
run_comprehensive_risk_analyst_tests()

📁 Added tests directory to Python path: c:\Users\sudip\Projects\courses\Udacity_Learning_Notes\AgenticAI_Finance\Project_TRACE\starter\tests
📊 Preview of Comprehensive Risk Analyst Tests:
   • Test RiskAnalystAgent initializes properly
   • Test handling of invalid JSON response
   • Test successful case analysis with valid response
   • Test OpenAI API call uses correct parameters
   • Test handling of empty LLM response
   ... and 5 more tests

💡 These tests validate edge cases you might not think of!
💡 Much more thorough than manual testing!
🧪 Comprehensive Risk Analyst Testing
📋 TODO: Uncomment and run after implementing your Risk Analyst Agent
🔍 Loading comprehensive test suite...
🚀 Running Risk Analyst test suite...
============================= test session starts =============================
platform win32 -- Python 3.11.14, pytest-9.1.1, pluggy-1.6.0 -- c:\Users\sudip\Projects\courses\Udacity_Learning_Notes\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\su

## ✅ Phase 3: Compliance Officer Agent Development

The Compliance Officer Agent uses **ReACT prompting** to generate regulatory-compliant SAR narratives.

### 📚 Understanding ReACT Prompting

ReACT (Reasoning + Action) prompting separates thinking and doing:

1. **Reasoning Phase**: Analyze situation and plan approach
2. **Action Phase**: Execute specific task with informed decisions
3. **Structured Workflow**: Consistent approach to complex tasks
4. **Regulatory Compliance**: Emphasis on meeting specific requirements

In [10]:

sys.path.append("..")
from src.compliance_officer_agent import create_react_framework

def design_react_prompt():
    """Design and test ReACT prompt for compliance narratives"""

    React_framework = create_react_framework()

    reasoning_phase = React_framework["reasoning_phase"]
    action_phase = React_framework["action_phase"]

    system_prompt = f"""
    You are a senior compliance officer responsible for generating
    regulatory-compliant SAR narratives.

    Use the following reasoning guidance internally:
    {reasoning_phase}

    Use the following narrative-generation guidance:
    {action_phase}

    REACT PROTOCOL:

    You operate in repeated THOUGHT → ACTION → OBSERVATION cycles.

    THOUGHT:
    Analyze what needs to be done next.

    ACTION:
    ACTION has one and only one meaning:
    request execution of one available tool.

    Every ACTION MUST consist of exactly one JSON tool request:

    ```json
    {{
        "tool": "<tool_name>",
        "parameters": {{...}}
    }}


    Do NOT put prose after ACTION.
    Do NOT output "DRAFT NARRATIVE".
    Do NOT describe what you intend to do as an ACTION.

    If you draft or revise a narrative, place that narrative inside the
    parameters of the appropriate validation tool call.

    AVAILABLE TOOLS:

    1. get_regulatory_requirements() - Check regulatory requirements
    3. _validate_narrative_compliance(self, narrative) - Validate that the narrative meets regulatory compliance requirements

    When you have a draft narrative, validate it by requesting:

    {{
        "tool": "_validate_narrative_compliance",
        "parameters": {{
            "narrative": "<draft narrative>"
        }}
    }}
    Wait for the OBSERVATION before continuing.

    If the validation result contains "valid": true:
    do not call another tool. Return FINAL immediately.

    FINAL:
    Return only a JSON object conforming to ComplianceOfficerOutput.

    Narrative requirements:
    - Maximum 120 words
    - Use objective regulatory language
    - Include supported dates, amounts, accounts and parties
    - Explain why the activity is suspicious
    - Do not invent facts or speculate
    """

    return system_prompt

# Testing the prompt design
react_prompt = design_react_prompt()
print("⚡ ReACT Prompt Design:")
print(react_prompt[:200] + "...")


⚡ ReACT Prompt Design:

    You are a senior compliance officer responsible for generating
    regulatory-compliant SAR narratives.

    Use the following reasoning guidance internally:
    ['Review risk analyst findings an...


In [11]:
# Implement and test Compliance Officer Agent - SIMPLE SMOKE TEST
# Students: Write a basic smoke test to verify your agent works
from openai import OpenAI
import sys
from pathlib import Path
import pandas as pd

# Notebook is located in starter/notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.risk_analyst_agent import RiskAnalystAgent
from src.compliance_officer_agent import ComplianceOfficerAgent

def create_compliance_sample_case(cust_id:str):
    # Load CSV data from ../data/
    customers_df, accounts_df, transactions_df = load_csv_data("../data/")

    # Create ExplainabilityLogger instance
    logger = ExplainabilityLogger("../outputs/audit_logs/smoke_test.jsonl")

    # Create DataLoader instance
    loader = DataLoader(logger)

    try:
        # Generate sample case
        sample_customer = customers_df[customers_df["customer_id"] == cust_id].copy()
        sample_customer['ssn_last_4'] = sample_customer['ssn_last_4'].astype(str)
        sample_account = accounts_df[accounts_df["customer_id"] == cust_id].copy()
        sample_transaction = transactions_df[transactions_df["account_id"].isin(sample_account["account_id"])].copy()
        sample_transaction["transaction_date"] = pd.to_datetime(
            sample_transaction["transaction_date"]
        ).dt.strftime("%Y-%m-%d")
        sample_transaction["counterparty"] = (
            sample_transaction["counterparty"].fillna("")
        )
        sample_transaction['location'] = sample_transaction['location'].astype(str)

        sample_customer_dict = sample_customer.to_dict(orient="records")[0]
        sample_account_dict = sample_account.to_dict(orient="records")
        sample_transaction_dict = sample_transaction.to_dict(orient="records")

        sample_case = loader.create_case_from_data(sample_customer_dict, sample_account_dict, sample_transaction_dict)
        return sample_case

    except Exception as e:
        print(f"Error generating sample case: {e}")
        return None

def simple_compliance_officer_smoke_test():
    """
    A simple smoke test for your Compliance Officer Agent
    
    This should be a basic test that:
    1. Creates a ComplianceOfficerAgent instance
    2. Creates simple test case and risk analysis data
    3. Calls generate_compliance_narrative() method  
    4. Verifies the result has a narrative with reasonable length
    5. Prints success/failure
    
    Keep this simple - just verify your agent doesn't crash and generates text!
    """

    print("✅ Compliance Officer Smoke Test")
    print("📋 Importing the ComplianceOfficerAgent")

    print("🧪 Creating simple test case and risk analysis data")
    logger = ExplainabilityLogger("../outputs/audit_logs/compliance_officer_smoke_test.jsonl")
    
    load_dotenv('../../.env')
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    try:
        print("📋 Creating Agent instances")
        risk_analyst_agent = RiskAnalystAgent(client, logger)
        compliance_officer_agent = ComplianceOfficerAgent(client, logger)
        # Create minimal test case
        compliance_test_case = create_compliance_sample_case("CUST_0002")
        risk_analysis_result = risk_analyst_agent.analyze_case(compliance_test_case)
        compliance_output = compliance_officer_agent.generate_compliance_narrative(compliance_test_case, risk_analysis_result)
        print(compliance_output)
        word_count = len(compliance_output.narrative.split())
        if word_count <= 120:
            print(f"✅ SUCCESS: Generated {word_count} word narrative")
            print(f"Preview: {compliance_output.narrative[:100]}...")
        else:
            print(f"❌ FAILED: Narrative too long ({word_count} words)")
    except Exception as e:
        print(f"❌ Compliance Officer Smoke test failed: {e}")

simple_compliance_officer_smoke_test()

✅ Compliance Officer Smoke Test
📋 Importing the ComplianceOfficerAgent
🧪 Creating simple test case and risk analysis data
📋 Creating Agent instances
🚀 STARTING CASE ANALYSIS

Finish reason: stop
Model: gpt-4.1-mini-2025-04-14

Usage: CompletionUsage(completion_tokens=660, prompt_tokens=1295, total_tokens=1955, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=1152, image_tokens=None, text_tokens=None))

========== RAW STEP 1 RESPONSE ==========
{'case_id': 'ac0340b8-2bf3-4916-9189-ffa011088827', 'customer': {'customer_id': 'CUST_0002', 'name': 'Renee Blair', 'risk_rating': <RiskAssessment.LOW: 'Low'>, 'occupation': 'Banker', 'annual_income': 72005, 'customer_since': '2023-03-03'}, 'accounts': [{'account_id': 'CUST_0002_ACC_1', 'account_type': 'Money_Market', 'status': 'Active',

### 🧪 Compliance Officer Testing Framework

In [12]:
# COMPREHENSIVE Compliance Officer Testing - Import Pre-Built Test Suite
# Students: Use our comprehensive test suite instead of writing your own

import sys
import os

# Add tests directory to Python path for importing test modules (if not already added)
project_root = os.path.abspath('..')
tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

print(f"📁 Tests directory in Python path: {tests_path}")

def run_comprehensive_compliance_officer_tests():
    """
    Use pre-built comprehensive test suite to validate your Compliance Officer Agent
    
    These tests validate:
    - Agent initialization and configuration
    - Narrative generation with valid inputs
    - Word count limits (≤120 words)
    - Regulatory citations inclusion
    - JSON parsing and error handling
    - ReACT prompt structure validation
    """
    print("🧪 Comprehensive Compliance Officer Testing")
    print("📋 TODO: Uncomment and run after implementing your Compliance Officer Agent")
    
    # Uncomment when your agent is ready:
    try:
        from test_compliance_officer import TestComplianceOfficerAgent
        import pytest
         
        print("📝 Loading comprehensive test suite...")
         
        # Run the test suite
        print("🚀 Running Compliance Officer test suite...")
        result = pytest.main([
            f"{tests_path}/test_compliance_officer.py", 
            "-v", 
            "--tb=short"
        ])
         
        if result == 0:
            print("✅ All Compliance Officer tests passed!")
        else:
            print("❌ Some tests failed. Check the output above for details.")
             
    except ImportError as e:
        print(f"❌ Import Error: {e}")
        print("💡 Make sure you've implemented ComplianceOfficerAgent in src/compliance_officer_agent.py")

# Quick preview of available tests
try:
    from test_compliance_officer import TestComplianceOfficerAgent
    import inspect
    
    # Get all test methods
    test_methods = [method for method in dir(TestComplianceOfficerAgent) 
                   if method.startswith('test_')]
    
    print("📝 Preview of Comprehensive Compliance Officer Tests:")
    for method_name in test_methods[:5]:  # Show first 5
        method = getattr(TestComplianceOfficerAgent, method_name)
        doc = method.__doc__ or method_name.replace('_', ' ').title()
        print(f"   • {doc}")
    if len(test_methods) > 5:
        print(f"   ... and {len(test_methods) - 5} more tests")
    print("\n💡 These tests validate regulatory compliance requirements!")
    print("💡 Includes word limits, citations, and required elements!")
except Exception as e:
    print(f"ℹ️ Test suite will be available after implementing foundation_sar.py: {e}")

# Call the function
run_comprehensive_compliance_officer_tests()

📁 Tests directory in Python path: c:\Users\sudip\Projects\courses\Udacity_Learning_Notes\AgenticAI_Finance\Project_TRACE\starter\tests
📝 Preview of Comprehensive Compliance Officer Tests:
   • Test ComplianceOfficerAgent initializes properly
   • Test OpenAI API call uses correct parameters
   • Test handling of empty LLM response
   • Test JSON extraction from code blocks
   • Test JSON extraction from plain text response
   ... and 5 more tests

💡 These tests validate regulatory compliance requirements!
💡 Includes word limits, citations, and required elements!
🧪 Comprehensive Compliance Officer Testing
📋 TODO: Uncomment and run after implementing your Compliance Officer Agent
📝 Loading comprehensive test suite...
🚀 Running Compliance Officer test suite...
============================= test session starts =============================
platform win32 -- Python 3.11.14, pytest-9.1.1, pluggy-1.6.0 -- c:\Users\sudip\Projects\courses\Udacity_Learning_Notes\.venv\Scripts\python.exe
cachedir

In [13]:
# COMPLETE AGENT TESTING - Two-Tier Approach
# Students: Use this to test both agents together

def complete_agent_testing_workflow():
    """
    Complete testing workflow using two-tier approach:
    
    TIER 1: Simple Smoke Tests (You write these)
    - Basic functionality verification
    - Quick sanity checks
    - Development debugging
    
    TIER 2: Comprehensive Test Suites (Pre-built for you)
    - Complex edge cases
    - Regulatory compliance validation
    - Professional-grade testing
    """
    print("🔬 Complete Agent Testing Workflow")
    print("=" * 50)
    
    print("\n📋 TIER 1: Simple Smoke Tests (DO FIRST)")
    print("   1. Write simple_risk_analyst_smoke_test() - verify basic functionality")
    print("   2. Write simple_compliance_officer_smoke_test() - verify basic functionality")
    print("   3. Fix any basic issues before moving to Tier 2")
    
    print("\n🧪 TIER 2: Comprehensive Test Suites (DO AFTER TIER 1 PASSES)")
    print("   1. Run comprehensive risk analyst test suite (10 comprehensive tests)")
    print("   2. Run comprehensive compliance officer test suite (10 comprehensive tests)")
    print("   3. Get detailed pass/fail results with specific feedback")
    
    print("\n💡 WHY THIS APPROACH?")
    print("   ✅ Tier 1: Quick feedback while developing")
    print("   ✅ Tier 2: Professional validation without writing complex tests")
    print("   ✅ Saves time: You focus on implementation, not test creation")
    print("   ✅ Better coverage: Our test suites test edge cases you might miss")

# Quick test runner when both agents are ready
def run_both_agents_quick_test():
    """Quick test of both agents using pre-built test suites"""
    print("🚀 Quick Test of Both Agents")
    print("📋 TODO: Uncomment when both agents are implemented")
    
    # Uncomment when ready:
    try:
        import pytest
         
        print("🔍 Running quick tests for both agents...")
         
        # Run a subset of tests for quick validation
        risk_result = pytest.main([
            f"{tests_path}/test_risk_analyst.py::TestRiskAnalystAgent::test_agent_initialization",
            f"{tests_path}/test_risk_analyst.py::TestRiskAnalystAgent::test_analyze_case_success",
            "-v"
        ])
         
        compliance_result = pytest.main([
            f"{tests_path}/test_compliance_officer.py::TestComplianceOfficerAgent::test_agent_initialization", 
            f"{tests_path}/test_compliance_officer.py::TestComplianceOfficerAgent::test_generate_compliance_narrative_success",
            "-v"
        ])
         
        if risk_result == 0 and compliance_result == 0:
            print("🎉 Both agents working! Ready for full test suite testing!")
        else:
            print("⚠️ Fix issues before running comprehensive tests")
            if risk_result != 0:
                print("   🔍 Risk Analyst needs fixes")
            if compliance_result != 0:
                print("   📝 Compliance Officer needs fixes")
                 
    except ImportError as e:
        print(f"❌ Import Error: {e}")
        print("💡 Make sure both agents are implemented")

complete_agent_testing_workflow()
run_both_agents_quick_test()

🔬 Complete Agent Testing Workflow

📋 TIER 1: Simple Smoke Tests (DO FIRST)
   1. Write simple_risk_analyst_smoke_test() - verify basic functionality
   2. Write simple_compliance_officer_smoke_test() - verify basic functionality
   3. Fix any basic issues before moving to Tier 2

🧪 TIER 2: Comprehensive Test Suites (DO AFTER TIER 1 PASSES)
   1. Run comprehensive risk analyst test suite (10 comprehensive tests)
   2. Run comprehensive compliance officer test suite (10 comprehensive tests)
   3. Get detailed pass/fail results with specific feedback

💡 WHY THIS APPROACH?
   ✅ Tier 1: Quick feedback while developing
   ✅ Tier 2: Professional validation without writing complex tests
   ✅ Saves time: You focus on implementation, not test creation
   ✅ Better coverage: Our test suites test edge cases you might miss
🚀 Quick Test of Both Agents
📋 TODO: Uncomment when both agents are implemented
🔍 Running quick tests for both agents...
============================= test session starts =========

## 🔗 Phase 4 Preview: Agent Integration

Once both agents are working, you'll integrate them into a complete workflow.

In [14]:
# TODO: Preview of integrated workflow
# This will be fully implemented in the next notebook

def preview_integrated_workflow():
    """Preview of how agents will work together"""
    
    workflow_steps = [
        "1. 📊 Load and validate case data",
        "2. 🔍 Risk Analyst performs Chain-of-Thought analysis",
        "3. 👤 Human review and approval gate",
        "4. ✅ Compliance Officer generates ReACT narrative (if approved)",
        "5. 📄 Generate complete SAR document",
        "6. 📊 Log audit trail and efficiency metrics"
    ]
    
    print("🔗 Integrated SAR Processing Workflow:")
    for step in workflow_steps:
        print(step)
    
    print("\n💡 Key Benefits:")
    print("• Two-stage processing reduces AI costs")
    print("• Human oversight ensures regulatory compliance")
    print("• Complete audit trails for examination")
    print("• Standardized analytical approaches")

preview_integrated_workflow()

🔗 Integrated SAR Processing Workflow:
1. 📊 Load and validate case data
2. 🔍 Risk Analyst performs Chain-of-Thought analysis
3. 👤 Human review and approval gate
4. ✅ Compliance Officer generates ReACT narrative (if approved)
5. 📄 Generate complete SAR document
6. 📊 Log audit trail and efficiency metrics

💡 Key Benefits:
• Two-stage processing reduces AI costs
• Human oversight ensures regulatory compliance
• Complete audit trails for examination
• Standardized analytical approaches


## 📝 Development Checklist - Two-Tier Testing Approach

### ✅ Risk Analyst Agent (Phase 2)
- [ ] Implement Chain-of-Thought system prompt
- [ ] Create `analyze_case` method with error handling
- [ ] Add JSON parsing and validation
- [ ] **TIER 1**: Write simple smoke test (verify basic functionality)
- [ ] **TIER 2**: Run comprehensive pre-built test suite (10 comprehensive tests)
- [ ] Fix any issues identified by test suite

### ✅ Compliance Officer Agent (Phase 3)  
- [ ] Implement ReACT system prompt
- [ ] Create `generate_compliance_narrative` method
- [ ] Add narrative validation (word count, terminology)
- [ ] **TIER 1**: Write simple smoke test (verify basic functionality)
- [ ] **TIER 2**: Run comprehensive pre-built test suite (10 comprehensive tests)
- [ ] Fix any issues identified by test suite

### ✅ Testing Strategy Benefits
- [ ] **Time Savings**: Focus on implementation, not complex test creation
- [ ] **Better Coverage**: Pre-built test suites test edge cases you might miss
- [ ] **Quick Feedback**: Simple smoke tests for rapid development cycles
- [ ] **Professional Validation**: Comprehensive test suites ensure production readiness
- [ ] **Regulatory Compliance**: Built-in checks for SAR requirements

### 💡 **Testing Workflow**
1. **Start with Tier 1**: Write simple smoke tests to verify your agents don't crash
2. **Fix basic issues**: Iterate quickly with simple tests during development
3. **Move to Tier 2**: Run comprehensive test suites when basic functionality works
4. **Analyze results**: Use detailed feedback to improve agent performance
5. **Iterate**: Refine prompts and logic based on test results

## 🚀 Next Steps

1. **Complete Agent Implementation**: Finish both agent classes in the src/ directory
2. **Run Two-Tier Testing**: Start with smoke tests, then comprehensive test suites
3. **Workflow Integration**: Move to the next notebook for complete system integration
4. **Human-in-the-Loop**: Implement decision gates and review processes

## 📊 Available Test Suites Summary

**Risk Analyst Test Suite (10 tests):**
- Agent initialization and configuration
- Case analysis with valid JSON responses
- JSON parsing and error handling
- System prompt structure validation
- API call parameter verification
- Helper method functionality
- Edge case handling

**Compliance Officer Test Suite (10 tests):**
- Agent initialization and configuration
- Narrative generation with valid responses
- Word count validation (≤120 words)
- Regulatory citations inclusion
- JSON parsing and error handling
- ReACT prompt structure validation
- API call parameter verification

**Ready to build intelligent agents with professional-grade testing! 🕵️‍♀️**